# Patient 1 

In this jupyter notebook the data of patient 1 is loaded, inspected and processed. 

In [1]:
# imports
import pickle
import mne
import numpy as np
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
import warnings
from toeplitzlda.classification import ToeplitzLDA

# utils functions
from utils.preprocessing import load_session_chached, merge_sessions
from utils.online_simulation import online_simulation
from utils.feature_extraction import load_features_chached
from utils.run_patient import run_patient_online_sessions

# Turn off warnings (that most likely occur from ToeplitzLDA)
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
mne.set_log_level('WARNING')



In [2]:
import line_profiler
%load_ext line_profiler

def my_function(x):
    e = 0
    for i in range(x):
        e+=i

# Example:
%lprun -f my_function my_function(500)

Timer unit: 1e-07 s

Total time: 0.0003431 s
File: C:\Users\Soz\AppData\Local\Temp\ipykernel_45256\559752206.py
Function: my_function at line 4

Line #      Hits         Time  Per Hit   % Time  Line Contents
     4                                           def my_function(x):
     5         1          7.0      7.0      0.2      e = 0
     6       501       1731.0      3.5     50.5      for i in range(x):
     7       500       1693.0      3.4     49.3          e+=i

In [ ]:

# features_online = load_features_chached(r"B:\_anonymized_data_P01a_P1_S4_anonymized.pkl")
# # for i in range(3,18):
# #     features = load_features_chached(f"B:\_anonymized_data_P01a_P1_S{i}_anonymized.pkl")

# #for i in range(3,14):
# #    features = load_features_chached(f"B:\_anonymized_data_P02a_P1_S{i}_anonymized.pkl")

# features_train = load_features_chached(r"B:\_anonymized_data_P01a_P1_S3_anonymized.pkl")

# from pathlib import Path
# import re

# def get_help_path_from_pickle_path(pickle_path):
#     """
#     Convert a flattened pickle filename like:
#         B:\_anonymized_data_P01a_P1_S1_anonymized.pkl
#     to the original folder-style path like:
#         B:\anonymized_data\P01a\P1_S1\anonymized
#     """
#     path = Path(pickle_path)
#     fname = path.name

#     match = re.match(r"_anonymized_data_(P\d+a)_(P\d+_S\d+)_anonymized\.pkl", fname)
#     if not match:
#         raise ValueError(f"Could not parse patient/session info from: {fname}")

#     part1 = match.group(1)  # e.g. P01a
#     part2 = match.group(2)  # e.g. P1_S1

#     help_path = path.parent / "anonymized_data" / part1 / part2 / "anonymized"
#     return str(help_path)
# from pathlib import Path
# import re

# import os

# def parse_pickle_path(pickle_path):
#     # Extract the full filename without directory
#     filename = os.path.basename(pickle_path)
    
#     # Regex to extract patient/session info
#     match = re.match(r'_anonymized_data_(P\d{2}a)_P(\d+)_S(\d+)_anonymized(.*)\.pkl', filename)
#     if not match:
#         raise ValueError(f"Invalid filename format: {filename}")
    
#     project, patient, session, suffix = match.groups()

#     # Construct help_path
#     help_path = os.path.join(
#         "B:/", "anonymized_data", project, f"P{patient}_S{session}", "anonymized"
#     )

#     # Extract optional selection and discard_channels from suffix
#     selection = None
#     discard_channels = False

#     if "_dc" in suffix:
#         discard_channels = True
#         suffix = suffix.replace("_dc", "")  # Remove _dc so it doesn't pollute selection

#     # Now match selection without _dc
#     sel_match = re.search(r'(6D_\w+)', suffix)
#     if sel_match:
#         selection = sel_match.group(1)

#     return help_path, selection, discard_channels

# import glob
# import os
# from utils.feature_extraction import load_or_extract_markers

# # Define the folder containing the data
# data_dir = r"B:\\"  # Ensure it ends with a backslash or double backslash

# # Use glob to find all .pkl files that match your pattern
# file_pattern = os.path.join(data_dir, "_anonymized_data*.pkl")
# all_pickle_files = glob.glob(file_pattern)

# print(f"Found {len(all_pickle_files)} .pkl files.")

# # Loop through all files and run your function
# for pickle_path in all_pickle_files:
#     print(f"\nProcessing pickle path: {pickle_path}")
#     #"B:\anonymized_data\P01a\P1_S1\anonymized"
#     help_path, selection, discard_channels = parse_pickle_path(pickle_path=pickle_path)
#     print("help_path:",help_path)  # B:\anonymized_data\P01a\P1_S1\anonymized
#     print("selection:",selection)  # 6D_long_350
#     print("dc:",discard_channels)  # True
#     online_trials = load_session_chached((help_path),selection=selection, discard_channels=discard_channels).get('trials')
#     try:
#         markers_info = load_or_extract_markers(pickle_path, online_trials)
#     except Exception as e:
#         print(f"Failed to process {pickle_path}: {e}")





In [ ]:
# features_s1 =load_features_chached(r"B:_anonymized_data_P01a_P1_S1_anonymized6D_long_350_dc.pkl")
# print(features_s1.keys())

# features_s2 =load_features_chached(r"B:_anonymized_data_P01a_P1_S2_anonymized6D_long_350_dc.pkl")
# print(features_s2.keys())

# features_merged = merge_features(features_s1, features_s2)


In [ ]:
# Run Transfer on all
from utils.db import patients_db

print(patients_db)
for id in patients_db:
    if id==8:
        info = patients_db.get(id)
        patient = info.get('patient_nr')
        last_session = info.get('last_session')
        calibration_selection = info.get('selection')

        print("patient: ", patient)
        print("last session", last_session)
        print("calibration_selection", calibration_selection)

        performances = run_patient_online_sessions(patient, last_session, calibration_selection)
        with open(f"p{patient}_performances_v1.pkl", 'wb') as f: 
            pickle.dump(performances, f)

In [ ]:
# Run static on all

from utils.static_protocol import static_protocol
from utils.run_patient import run_patient_online_sessions_static

print(static_protocol)
for id in static_protocol:
    info = static_protocol.get(id)
    patient = info.get('patient_nr')
    last_session = info.get('last_session')
    calibration_selection = info.get('selection')
    changing_conditions = info.get('changing_condition')
    if changing_conditions:
        starter_conditions = info.get('changing_starter_sessions')
    else:
        starter_conditions = None
        
    print("patient: ", patient)
    print("last session: ", last_session)
    print("calibration_selection: ", calibration_selection)
    print("changing conditions: ", changing_conditions)

    performances = run_patient_online_sessions_static(patient, last_session, calibration_selection, starter_conditions)
    with open(f"p{patient}_performances_static_v1.pkl", 'wb') as f: 
        pickle.dump(performances, f)